In [1]:
from transformers import AutoModel, AutoTokenizer
import time
import json
import torch
from pathlib import Path

### Model loading

In [2]:
start = time.time()
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# float16 causes placeholder tensor issues on MPS; use float32 on MPS, float16 on CUDA only
dtype = torch.float16 if device.type == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(
    "jinaai/jina-embeddings-v5-text-small", trust_remote_code=True
)
model = AutoModel.from_pretrained(
    "jinaai/jina-embeddings-v5-text-small",
    trust_remote_code=True,
    torch_dtype=dtype,
).to(device)

print("Using device:", device)
print(f"Model loaded in {time.time() - start:.2f}s")
print(f"Model params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Using device: mps
Model loaded in 10.78s
Model params: 676.8M


In [3]:
texts = [
    "My order hasn't arrived yet and it's been two weeks.",
    "How do I reset my password?",
    "I'd like a refund for my recent purchase.",
    "Your product exceeded my expectations. Great job!",
]
classification_embeddings = model.encode(texts=texts, task="classification")

In [4]:
print(classification_embeddings)

tensor([[ 0.0214, -0.0469, -0.0241,  ...,  0.0207, -0.0150, -0.0066],
        [ 0.0153, -0.0446, -0.0451,  ..., -0.0203, -0.0198, -0.0042],
        [ 0.0578, -0.0324, -0.0183,  ...,  0.0153, -0.0141,  0.0125],
        [-0.0386, -0.0292, -0.0375,  ..., -0.0054, -0.0190,  0.0115]],
       device='mps:0')


In [5]:
def get_embeddings(texts: list[str], batch_size: int = 8) -> torch.Tensor:
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=512, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output = model(**encoded)

        mask = encoded["attention_mask"].unsqueeze(-1).float()
        pooled = (output.last_hidden_state * mask).sum(1) / mask.sum(1)

        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())

    return torch.cat(all_embeddings, dim=0)

In [6]:
human_texts = [
    "honestly i have no idea what i'm doing but somehow it's working lol",
    "my coffee got cold again because i forgot about it, third time this week",
    "just got back from the gym, legs are completely dead but i feel great",
    "been thinking about switching jobs but the market rn is kinda rough tbh",
    "my cat knocked over my coffee this morning and i just stared at the mess",
    "cant believe how fast this year went, feels like january was last week",
    "bro why is my wifi only slow when i actually need it the most",
    "i was so tired i put the milk in the cabinet and cereal in the fridge",
    "woke up at 3am randomly and now i'm just scrolling through my phone",
    "the universe really said 'let's make this persons day worse' and i felt that",
]

ai_texts = [
    "The implementation of advanced machine learning algorithms facilitates enhanced operational efficiency.",
    "Natural language processing has advanced considerably, enabling models to generate coherent text.",
    "It is important to consider the ethical implications of deploying AI systems in sensitive domains.",
    "Organizations must establish robust governance frameworks to ensure responsible AI development.",
    "The utilization of advanced computational methods facilitates enhanced operational efficiency.",
    "In conclusion, leveraging AI technologies can substantially improve organizational performance metrics.",
    "This comprehensive analysis examines the multifaceted implications of artificial intelligence adoption.",
    "Furthermore, the integration of machine learning models requires careful consideration of data quality.",
    "The proliferation of large language models represents a paradigm shift in human-computer interaction.",
    "Stakeholders must collaboratively address the societal challenges posed by generative AI technologies.",
]

In [7]:
from sklearn.linear_model import LogisticRegression


train_texts = human_texts + ai_texts
train_labels = [0] * len(human_texts) + [1] * len(ai_texts)

train_embeddings = get_embeddings(train_texts)

clf = LogisticRegression(max_iter=1000)
clf.fit(train_embeddings, train_labels)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [8]:
def predict(text: str):
    emb = get_embeddings([text])
    prob = clf.predict_proba(emb)[0]
    ai_prob = prob[1] * 100
    human_prob = prob[0] * 100

    label = "AI-written" if prob[1] >= 0.5 else "Human-written"

    bar_len = 30
    ai_bar = "█" * int(ai_prob / 100 * bar_len)
    human_bar = "█" * int(human_prob / 100 * bar_len)

    print("\n")
    if label == "AI-written":
        print("Verdict: AI generated")
    else:
        print("Verdict: Human written")

    print(f"Confidence: {max(ai_prob, human_prob):.1f} %")
    print(f"\nAI probability: {ai_bar:<30} {ai_prob:.1f} %")
    print(f"\nHuman probability: {human_bar:<30} {human_prob:.1f} %\n\n")

    return ai_prob, human_prob

In [9]:
LOG_PATH = Path("predictions_log.json")


def save_result(record: dict):
    """Append prediction record to JSON file safely."""
    if LOG_PATH.exists():
        with open(LOG_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = []

    data.append(record)

    with open(LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

#### data for inference

In [ ]:
human_texts = [
    "ordered pizza and they forgot my drink, how do you even forget the drink man",
    "my brain at 2am: remember that embarrassing thing from 7 years ago? let's think about that",
    "told myself id go to bed early tonight and here i am watching tiktoks at 1am",
    "just spent 20 minutes looking for my phone while i was holding it lol kill me",
    "the way my heart drops when i see 'we need to talk' message oh no",
    "why does food taste better when someone else makes it like what's the science behind that",
    "me: i need to save money also me: *adds random stuff to amazon cart at midnight*",
    "when the cashier says 'have a nice day' and i say 'you too' like they dont have a choice now",
    "my phone battery at 1% and im still scrolling like i'm playing a dangerous game",
    "that moment when you realize tomorrow is monday and you did nothing all weekend classic",
    "accidentally liked someone's photo from 2018 while stalking their profile time to delete the app",
    "why is it that every time i wash my car it rains immediately like who do i gotta talk to",
    "me trying to explain to my parents what my job is: so there's this thing called the cloud...",
    "the audacity of my alarm clock to go off right when im having a good dream",
    "when someone says 'can i ask you a question' and my brain instantly assumes i'm in trouble",
    "i put a reminder on my phone to check my reminders and now i'm stuck in a loop",
    "that feeling when you wave at someone but they didnt see you so you just pretend to fix your hair",
    "my sleep paralysis demon watching me scroll through twitter at 4am: pathetic",
    "went to the store for milk and came back with everything except milk how does this keep happening",
    "the way i rehearse conversations in the shower and then say none of it in real life",
]

ai_texts = [
    "The integration of artificial intelligence into daily life continues to accelerate at an unprecedented pace.",
    "Machine learning models require substantial amounts of high-quality training data to achieve optimal performance.",
    "It is recommended that users regularly update their software to maintain security and functionality.",
    "The correlation between sleep deprivation and reduced cognitive function is well-documented in scientific literature.",
    "E-commerce platforms utilize sophisticated algorithms to personalize product recommendations for individual users.",
    "Understanding the fundamental principles of neural networks is essential for aspiring AI researchers.",
    "The global supply chain faces numerous challenges including labor shortages and logistical constraints.",
    "Effective time management strategies can significantly improve both professional productivity and personal well-being.",
    "Recent studies indicate that remote work arrangements may increase job satisfaction among certain demographics.",
    "The development of autonomous vehicles raises important questions about liability and regulatory frameworks.",
    "Financial advisors generally recommend diversifying investment portfolios to mitigate potential market volatility.",
    "The environmental impact of cryptocurrency mining has become a topic of increasing concern among policymakers.",
    "Educational institutions are gradually incorporating digital literacy skills into their core curricula.",
    "Advancements in natural language processing enable more sophisticated human-computer interaction paradigms.",
    "The healthcare sector continues to explore AI applications for diagnostic imaging and patient care optimization.",
    "Sustainable practices in manufacturing can reduce operational costs while minimizing ecological footprint.",
    "Consumer behavior patterns have shifted significantly in response to global economic uncertainties.",
    "The legal implications of data privacy violations can result in substantial penalties for organizations.",
    "Research suggests that mindfulness practices may contribute to improved mental health outcomes.",
    "Technological innovation remains a primary driver of economic growth in developed nations.",
]

In [13]:
import time
import psutil
import torch
from datetime import datetime

# Optional GPU
try:
    from pynvml import *

    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(0)
except:
    handle = None


def run_inference(text):
    """
    Run AI detector inference and collect only the metrics you want.
    """
    start_time = time.perf_counter()
    ai_prob, human_prob = predict(text)
    elapsed = time.perf_counter() - start_time

    # CPU & RAM
    cpu_usage = psutil.cpu_percent(interval=None)
    ram_usage = psutil.virtual_memory().percent

    # GPU (CUDA)
    gpu_util = None
    gpu_mem = None
    if torch.cuda.is_available() and handle:
        util = nvmlDeviceGetUtilizationRates(handle)
        mem = nvmlDeviceGetMemoryInfo(handle)
        gpu_util = util.gpu
        gpu_mem = mem.used / 1024**2

    record = {
        "timestamp": datetime.utcnow().isoformat(),
        "text": text,
        "ai_probability": float(ai_prob),
        "human_probability": float(human_prob),
        "inference_time_sec": round(elapsed, 4),
        "cpu_usage_percent": cpu_usage,
        "ram_usage_percent": ram_usage,
        "gpu_usage_percent": gpu_util,
        "gpu_memory_mb": gpu_mem,
    }

    save_result(record)
    print(f"AI: {ai_prob:.2f}% | Human: {human_prob:.2f}% | Time: {elapsed:.3f}s")

    return record


def evaluate_texts(human_texts, ai_texts):
    results = []

    print("\nTesting Human texts: \n")
    for text in human_texts:
        results.append(run_inference(text))

    print("\nTesting AI texts: \n")
    for text in ai_texts:
        results.append(run_inference(text))

    return results


results = evaluate_texts(human_texts, ai_texts)


Testing Human texts: 



Verdict: Human written
Confidence: 56.6 %

AI probability: █████████████                  43.4 %

Human probability: ████████████████               56.6 %


AI: 43.44% | Human: 56.56% | Time: 0.579s


Verdict: Human written
Confidence: 53.6 %

AI probability: █████████████                  46.4 %

Human probability: ████████████████               53.6 %


AI: 46.44% | Human: 53.56% | Time: 0.036s


Verdict: Human written
Confidence: 52.6 %

AI probability: ██████████████                 47.4 %

Human probability: ███████████████                52.6 %


AI: 47.39% | Human: 52.61% | Time: 0.031s


/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),




Verdict: Human written
Confidence: 54.2 %

AI probability: █████████████                  45.8 %

Human probability: ████████████████               54.2 %


AI: 45.82% | Human: 54.18% | Time: 0.475s


Verdict: Human written
Confidence: 55.2 %

AI probability: █████████████                  44.8 %

Human probability: ████████████████               55.2 %


AI: 44.77% | Human: 55.23% | Time: 0.034s


Verdict: Human written
Confidence: 57.4 %

AI probability: ████████████                   42.6 %

Human probability: █████████████████              57.4 %


AI: 42.63% | Human: 57.37% | Time: 0.030s


Verdict: Human written
Confidence: 59.4 %

AI probability: ████████████                   40.6 %

Human probability: █████████████████              59.4 %


AI: 40.58% | Human: 59.42% | Time: 0.028s


Verdict: Human written
Confidence: 60.2 %

AI probability: ███████████                    39.8 %

Human probability: ██████████████████             60.2 %


AI: 39.81% | Human: 60.19% | Time: 0.

/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/var



Verdict: AI generated
Confidence: 57.9 %

AI probability: █████████████████              57.9 %

Human probability: ████████████                   42.1 %


AI: 57.89% | Human: 42.11% | Time: 0.422s


Verdict: AI generated
Confidence: 57.4 %

AI probability: █████████████████              57.4 %

Human probability: ████████████                   42.6 %


AI: 57.41% | Human: 42.59% | Time: 0.029s


Verdict: AI generated
Confidence: 54.5 %

AI probability: ████████████████               54.5 %

Human probability: █████████████                  45.5 %


AI: 54.46% | Human: 45.54% | Time: 0.023s


/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),




Verdict: AI generated
Confidence: 55.9 %

AI probability: ████████████████               55.9 %

Human probability: █████████████                  44.1 %


AI: 55.85% | Human: 44.15% | Time: 0.410s


/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),




Verdict: AI generated
Confidence: 56.3 %

AI probability: ████████████████               56.3 %

Human probability: █████████████                  43.7 %


AI: 56.33% | Human: 43.67% | Time: 0.414s


Verdict: AI generated
Confidence: 57.8 %

AI probability: █████████████████              57.8 %

Human probability: ████████████                   42.2 %


AI: 57.76% | Human: 42.24% | Time: 0.027s


Verdict: AI generated
Confidence: 54.2 %

AI probability: ████████████████               54.2 %

Human probability: █████████████                  45.8 %


AI: 54.24% | Human: 45.76% | Time: 0.029s


Verdict: AI generated
Confidence: 55.4 %

AI probability: ████████████████               55.4 %

Human probability: █████████████                  44.6 %


AI: 55.42% | Human: 44.58% | Time: 0.031s


Verdict: AI generated
Confidence: 59.8 %

AI probability: █████████████████              59.8 %

Human probability: ████████████                   40.2 %


AI: 59.75% | Human: 40.25% | Time: 0.024s


/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_11010/1426766967.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/var